# CLIWOC Data Cleaning and Preparation


**Output files written to `data/`:**
- `cliwoc_clean.csv`
- `routes_sample.geojson`
- `nation_stats.json`
- `wind_words_by_nation.json`
- `encounters.json`

## Setup

In [1]:
import json
import os
import re
import sys
from collections import defaultdict

import numpy as np
import pandas as pd

np.random.seed(42)

## Paths & constants

In [2]:
NOTEBOOK_DIR = os.path.abspath('')

RAW_PATH = os.path.abspath(os.path.join(NOTEBOOK_DIR, 'CLIWOC21.tsv'))


CLEAN_CSV         = os.path.join(NOTEBOOK_DIR, 'cliwoc_clean.csv')
ROUTES_GEOJSON    = os.path.join(NOTEBOOK_DIR, 'routes_sample.geojson')
NATION_STATS_JSON = os.path.join(NOTEBOOK_DIR, 'nation_stats.json')
WIND_WORDS_JSON   = os.path.join(NOTEBOOK_DIR, 'wind_words_by_nation.json')
ENCOUNTERS_JSON   = os.path.join(NOTEBOOK_DIR, 'encounters.json')

YEAR_MIN    = 1750
YEAR_MAX    = 1850
MAIN_NATIONS = {'British', 'Dutch', 'French', 'Spanish'}

print('Raw data path :', RAW_PATH or '*** NOT FOUND — see note below ***')

Raw data path : /run/media/christian/64609cdd-60c1-4856-8535-f3f0c6d7375c/home/christian/OneDrive/DTU/02806 Social Data Analysis and Visualization/winds-of-empire/data/CLIWOC21.tsv


In [3]:
NATIONALITY_MAP = {
    # --- BRITISH ---
    'English': 'British', 'english': 'British', 'ENGLISH': 'British',
    'british': 'British', 'BRITISH': 'British', 'BRITITSH': 'British',
    'BRISTISH': 'British', 'BRITSH': 'British', 'BRITI': 'British',
    'BRITISH(?)': 'British', 'BRITISH ?': 'British', 'BRITISH?': 'British',
    'BRITISH (?)': 'British', 'BRITISH (RN)': 'British', 'BRITSH AND SPANISH': 'British',
    '2 SHIPS-BRITISH(?)': 'British', 'BRITISH AND OTHER': 'British',
    'BRITISH AND OTHERS': 'British', 'BRITISH/FRENCH': 'British',
    'BRITISH/DUTCH': 'British', 'BRITISH AND AMERICAN': 'British',

    # --- DUTCH ---
    'Nederlands': 'Dutch', 'netherlands': 'Dutch', 'NETHERLANDS': 'Dutch',
    'dutch': 'Dutch', 'DUTCH': 'Dutch', 'Holland': 'Dutch', 'hollands': 'Dutch',
    'DUTCH (ZEELAND)': 'Dutch', 'DUTC': 'Dutch', 'UNKNOWN/DUTCH': 'Dutch',
    'PRINSEVLAG': 'Dutch', 'DUTCH AND BRITISH': 'Dutch', 'DUTCH AND UNKNOWN': 'Dutch',

    # --- SPANISH ---
    'spanish': 'Spanish', 'SPANISH': 'Spanish', 'ESPANOL': 'Spanish',
    'Espanol': 'Spanish', 'CATALANA': 'Spanish',

    # --- FRENCH ---
    'french': 'French', 'FRENCH': 'French', 'FRANCAIS': 'French',

    # --- AMERICAN ---
    'AMERICAN': 'American', 'NORTH AMERICAN': 'American', 'USA': 'American',

    # --- PORTUGUESE ---
    'PORTUGUESE': 'Portuguese', 'PORTUGESE': 'Portuguese',

    # --- DANISH ---
    'DANISH': 'Danish', 'DAMISH': 'Danish',

    # --- SWEDISH ---
    'SWEDISH': 'Swedish',

    # --- RUSSIAN ---
    'RUSSIAN': 'Russian',

    # --- PRUSSIAN ---
    'PRUSIAN': 'Prussian', 'PRUSSIAN': 'Prussian',

    # --- GERMAN ---
    'GERMAN': 'German', 'HAMBURGIA': 'German', 'HAMBURG': 'German',

    # --- NORWEGIAN ---
    'NORWEGIAN': 'Norwegian', 'NORWAY': 'Norwegian',

    # --- ITALIAN ---
    'GENOESE': 'Italian', 'VENICETIAN AND BRITISH': 'Italian',
    'VENETIANS': 'Italian', 'SICILIAN': 'Italian',

    # --- UNKNOWN & OTHERS ---
    'UNKNOWN': 'Unknown', 'UNKNOWN ANGRAKA': 'Unknown', 'UKNOWN': 'Unknown',
}

_STOPWORDS = {
    'the', 'a', 'an', 'and', 'or', 'of', 'with', 'at', 'to', 'in',
    'is', 'was', 'it', 'its', 'by', 'on', 'from', 'had', 'but', 'no',
    'not', 'be', 'very', 'some', 'all', 'for', 'as', 'se', 'de', 'la',
    'le', 'et', 'du', 'van', 'een', 'met', 'en', 'het', 'op', 'te',
    'y', 'el', 'los', 'las', 'con', 'que', 'por', 'del', 'un', 'una',
    'zijn', 'was', 'heeft', 'naar', 'uit', 'over', 'aan', 'bij',
}

## Step 1.1 — Load raw data

In [4]:
print(f'[1.1] Loading raw data from:\n      {RAW_PATH}')
sep = '\t' if RAW_PATH.endswith('.tsv') else ','
df_raw = pd.read_csv(RAW_PATH, sep=sep, low_memory=False, encoding='latin-1')
print(f'      Shape: {df_raw.shape}')
print(f'      Columns ({len(df_raw.columns)}): {list(df_raw.columns)[:20]} ...')
df_raw.head(3)

[1.1] Loading raw data from:
      /run/media/christian/64609cdd-60c1-4856-8535-f3f0c6d7375c/home/christian/OneDrive/DTU/02806 Social Data Analysis and Visualization/winds-of-empire/data/CLIWOC21.tsv
      Shape: (287116, 180)
      Columns (180): ['YR', 'MO', 'DY', 'HR', 'LAT', 'LON', 'latitude', 'longitude', 'IM', 'ATTC', 'TI', 'LI', 'DS', 'VS', 'NID', 'II', 'ID', 'C1', 'DI', 'D'] ...


,YR,MO,DY,HR,LAT,LON,latitude,longitude,IM,ATTC,...,CloudFrac,Gusts,Rain,Fog,Snow,Thunder,Hail,SeaIce,TrivialCorrection,Release
0,1827,8,16,500,-692,10310,-6.92,103.1,0,1,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,CLIWOC VERSION 1.0
1,1827,8,17,500,-827,10047,-8.27,100.47,0,1,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,CLIWOC VERSION 1.0
2,1827,8,18,500,-943,9772,-9.43,97.72,0,1,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,CLIWOC VERSION 1.0


## Step 1.2 — Core cleaning

In [5]:
def safe_int(s, default=1):
    try:
        return int(s)
    except (ValueError, TypeError):
        return default


def make_date(row):
    try:
        return pd.Timestamp(
            year=int(row['year_int']),
            month=max(1, min(12, int(row['month_int']))),
            day=max(1, min(28, int(row['day_int']))),
        )
    except Exception:
        return pd.NaT

In [6]:
print('[1.2] Core cleaning ...')
orig = len(df_raw)
df = df_raw.copy()

# Parse dates
df['year_int']  = df['Year'].apply(lambda x: safe_int(x, 0))
df['month_int'] = df['Month'].apply(lambda x: safe_int(x, 1))
df['day_int']   = df['Day'].apply(lambda x: safe_int(x, 1))
df['date']      = df.apply(make_date, axis=1)
print(f'      Year range in raw data: {df["year_int"].min()}-{df["year_int"].max()}  ({orig:,} rows)')

# Year filter
df = df[(df['year_int'] >= YEAR_MIN) & (df['year_int'] <= YEAR_MAX)].copy()
print(f'      After year filter ({YEAR_MIN}-{YEAR_MAX}):       {len(df):,} rows')

# Nationality standardise & filter
if 'Nationality' not in df.columns:
    for alt in ['nationality', 'Nation', 'nation', 'Country']:
        if alt in df.columns:
            df.rename(columns={alt: 'Nationality'}, inplace=True)
            break

df['Nationality'] = df['Nationality'].replace(NATIONALITY_MAP)
df = df[df['Nationality'].isin(MAIN_NATIONS)].copy()
print(f'      After nationality filter (4 nations):   {len(df):,} rows')
print(df['Nationality'].value_counts().to_string())

# Coordinate filter
lat_col = 'latitude'
lon_col = 'longitude'
df.rename(columns={lat_col: 'Lat', lon_col: 'Lon'}, inplace=True)
df['Lat'] = pd.to_numeric(df['Lat'], errors='coerce')
df['Lon'] = pd.to_numeric(df['Lon'], errors='coerce')
before_coord = len(df)
df = df[~(df['Lat'].isna() & df['Lon'].isna())].copy()
print(f'      After coord filter: {len(df):,} rows  (removed {before_coord - len(df):,})')

df_clean = df
print('\nCleaning complete.')

[1.2] Core cleaning ...
      Year range in raw data: 0-1855  (287,116 rows)
      After year filter (1750-1850):       272,562 rows
      After nationality filter (4 nations):   271,778 rows
Nationality
Dutch      112610
British     94859
Spanish     53824
French      10485
      After coord filter: 251,456 rows  (removed 20,322)

Cleaning complete.


## Step 1.3 — Wind & Beaufort features

In [7]:
print('[1.3] Wind & Beaufort features ...')

# Numeric Beaufort column
bft_col = 'W'
df_clean = df_clean.copy()
df_clean.rename(columns={bft_col: 'Bft'}, inplace=True)
df_clean['Bft'] = pd.to_numeric(df_clean['Bft'], errors='coerce')
# W is stored in tenths of m/s (e.g. 67 = 6.7 m/s = Bft 4).
# Cap at 12 was wrong — it kept only <=1.2 m/s (Bft 0-1) and discarded ~90% of observations.
# Convert to Beaufort class using WMO-1100 boundaries (in tenths of m/s).
_bft_bins = [0, 2, 15, 33, 54, 79, 107, 138, 171, 207, 244, 284, 326, float('inf')]
df_clean['Bft'] = pd.cut(df_clean['Bft'], bins=_bft_bins, labels=range(13),
                             include_lowest=True).astype(float)

print(f'      Bft coverage: {df_clean["Bft"].notna().sum():,} non-null')
print(f'      Bft distribution:\n{df_clean["Bft"].value_counts().sort_index().to_string()}')

df_clean['beaufort_era'] = np.where(df_clean['year_int'] >= 1806, 'Beaufort', 'pre-Beaufort')
print(df_clean['beaufort_era'].value_counts().to_string())

# Free-text wind description
wind_text_cols = [
    c for c in df_clean.columns
    if any(kw in c.lower() for kw in ['wind', 'weather', 'allwind', 'referencewind', 'statese'])
    and df_clean[c].dtype == object
]
print(f'      Wind-text columns: {wind_text_cols}')

def concat_wind(row):
    parts = []
    for c in wind_text_cols:
        val = row.get(c, '')
        if isinstance(val, str) and val.strip():
            parts.append(val.strip())
    return ' '.join(parts) if parts else np.nan

df_clean['wind_description'] = df_clean.apply(concat_wind, axis=1)
non_null = df_clean['wind_description'].notna().sum()
print(f'      Rows with wind_description: {non_null:,} ({non_null/len(df_clean)*100:.1f}%)')

[1.3] Wind & Beaufort features ...
      Bft coverage: 217,561 non-null
      Bft distribution:
Bft
0.0      6784
1.0     18107
2.0     32840
3.0     32613
4.0     49748
5.0     37304
6.0     18113
7.0      6065
8.0     10669
9.0      3745
10.0     1336
11.0      210
12.0       27
beaufort_era
pre-Beaufort    167328
Beaufort         84128
      Wind-text columns: []
      Rows with wind_description: 0 (0.0%)


In [8]:
# Word frequency per nationality → wind_words_by_nation.json
word_freq = {}
for nation in sorted(MAIN_NATIONS):
    sub = df_clean[df_clean['Nationality'] == nation]['wind_description'].dropna()
    freq = defaultdict(int)
    for text in sub:
        for w in re.findall(r'[a-zA-ZÀ-ÿ]+', str(text).lower()):
            if w not in _STOPWORDS and len(w) > 2:
                freq[w] += 1
    top = dict(sorted(freq.items(), key=lambda x: -x[1])[:100])
    word_freq[nation] = top
    print(f'      {nation}: top words = {list(top.items())[:5]}')

with open(WIND_WORDS_JSON, 'w') as f:
    json.dump(word_freq, f)
print(f'\nSaved: {WIND_WORDS_JSON}')

      British: top words = []
      Dutch: top words = []
      French: top words = []
      Spanish: top words = []

Saved: /run/media/christian/64609cdd-60c1-4856-8535-f3f0c6d7375c/home/christian/OneDrive/DTU/02806 Social Data Analysis and Visualization/winds-of-empire/data/wind_words_by_nation.json


## Step 1.4 — Route segments (GeoJSON)

In [9]:
MAX_SEGMENTS = 5000
print(f'[1.4] Building route segments (max {MAX_SEGMENTS}) ...')

df_clean['unique_voyage_id'] = df_clean['ShipName'].astype(str) + "_" + df_clean['VoyageIni'].astype(str)
voyage_col = 'unique_voyage_id'

df_sorted = df_clean.dropna(subset=['Lat', 'Lon']).sort_values(
    [voyage_col, 'year_int', 'month_int', 'day_int']
)

nation_counts = df_sorted['Nationality'].value_counts()
total = nation_counts.sum()
quota = {n: max(1, int(MAX_SEGMENTS * c / total)) for n, c in nation_counts.items()}

features = []
for nation in sorted(MAIN_NATIONS):
    if nation not in nation_counts.index:
        continue
    sub = df_sorted[df_sorted['Nationality'] == nation]
    segs = []
    for vid, grp in sub.groupby(voyage_col):
        coords = list(zip(grp['Lon'].tolist(), grp['Lat'].tolist()))
        for i in range(len(coords) - 1):
            if abs(coords[i][0] - coords[i+1][0]) > 60 or \
               abs(coords[i][1] - coords[i+1][1]) > 60:
                continue
            segs.append({
                'type': 'Feature',
                'geometry': {
                    'type': 'LineString',
                    'coordinates': [
                        [round(coords[i][0], 4),   round(coords[i][1], 4)],
                        [round(coords[i+1][0], 4), round(coords[i+1][1], 4)],
                    ],
                },
                'properties': {
                    'nationality': nation,
                    'year': int(grp['year_int'].iloc[0]),
                },
            })
    n_keep = min(quota.get(nation, 500), len(segs))
    if n_keep < len(segs):
        idx = np.random.choice(len(segs), n_keep, replace=False)
        segs = [segs[i] for i in idx]
    features.extend(segs)
    print(f'      {nation}: {len(segs):,} segments')

geojson = {'type': 'FeatureCollection', 'features': features}
with open(ROUTES_GEOJSON, 'w') as f:
    json.dump(geojson, f)
print(f'\nSaved: {ROUTES_GEOJSON}  ({len(features):,} total segments)')

[1.4] Building route segments (max 5000) ...
      British: 1,844 segments
      Dutch: 2,072 segments
      French: 166 segments
      Spanish: 915 segments

Saved: /run/media/christian/64609cdd-60c1-4856-8535-f3f0c6d7375c/home/christian/OneDrive/DTU/02806 Social Data Analysis and Visualization/winds-of-empire/data/routes_sample.geojson  (4,997 total segments)


## Step 1.5 — Nation-year aggregations

In [10]:
print('[1.5] Nation-year aggregations ...')

records = []
for (nation, year), grp in df_clean.groupby(['Nationality', 'year_int']):
    avg_bft = grp['Bft'].mean() if 'Bft' in grp.columns else np.nan
    wind_cov = grp['wind_description'].notna().mean() \
        if 'wind_description' in grp.columns else 0.0
    avg_bft_val = None \
        if (isinstance(avg_bft, float) and np.isnan(avg_bft)) \
        else round(float(avg_bft), 3)
    records.append({
        'nation':        nation,
        'year':          int(year),
        'count':         int(len(grp)),
        'avg_bft':       avg_bft_val,
        'wind_coverage': round(float(wind_cov), 4),
    })

with open(NATION_STATS_JSON, 'w') as f:
    json.dump(records, f)
print(f'Saved: {NATION_STATS_JSON}  ({len(records)} records)')
pd.DataFrame(records).head(8)

[1.5] Nation-year aggregations ...
Saved: /run/media/christian/64609cdd-60c1-4856-8535-f3f0c6d7375c/home/christian/OneDrive/DTU/02806 Social Data Analysis and Visualization/winds-of-empire/data/nation_stats.json  (271 records)


,nation,year,count,avg_bft,wind_coverage
0,British,1750,2024,4.652,0.0
1,British,1751,1497,4.598,0.0
2,British,1752,1485,4.408,0.0
3,British,1753,1001,4.423,0.0
4,British,1754,1038,4.716,0.0
5,British,1755,1201,5.335,0.0
6,British,1756,995,5.202,0.0
7,British,1757,1139,5.083,0.0


## Step 1.6 — Ship encounter analysis

In [11]:
print('[1.6] Ship encounter analysis ...')

enc_cols = [
    c for c in df_clean.columns
    if any(kw in c.lower() for kw in ['encnat', 'encname', 'encrem'])
]
print(f'      Encounter-related columns found: {enc_cols}')

edges = defaultdict(int)

enc_nat_col = next((c for c in enc_cols if 'encnat' in c.lower()), None)
if enc_nat_col:
    sub = df_clean[
        df_clean[enc_nat_col].notna() &
        (df_clean[enc_nat_col].astype(str).str.strip() != '')
    ]
    if len(sub) > 0:
        print(f"      Using real encounter column '{enc_nat_col}' ({len(sub):,} rows).")
        synthetic = False
        for _, row in sub.iterrows():
            my_nat = NATIONALITY_MAP.get(
                str(row['Nationality']).strip(), str(row['Nationality']).strip()
            )
            for met_raw in str(row[enc_nat_col]).split(','):
                met_val = NATIONALITY_MAP.get(met_raw.strip(), met_raw.strip())
                if met_val and my_nat:
                    key = tuple(sorted([my_nat, met_val]))
                    edges[key] += 1
    else:
        print('      WARNING: No real encounter data found in the EncNat column!')

edge_list = [
    {'nation_a': k[0], 'nation_b': k[1], 'count': v}
    for k, v in edges.items()
]
edge_list.sort(key=lambda x: -x['count'])

result = {'edges': edge_list}
with open(ENCOUNTERS_JSON, 'w') as f:
    json.dump(result, f)
print(f'Saved: {ENCOUNTERS_JSON}  ({len(edge_list)} unique nation-pairs found)')
pd.DataFrame(edge_list).head(8)

[1.6] Ship encounter analysis ...
      Encounter-related columns found: ['EncName', 'EncNat', 'EncRem']
      Using real encounter column 'EncNat' (30,070 rows).
Saved: /run/media/christian/64609cdd-60c1-4856-8535-f3f0c6d7375c/home/christian/OneDrive/DTU/02806 Social Data Analysis and Visualization/winds-of-empire/data/encounters.json  (105 unique nation-pairs found)


,nation_a,nation_b,count
0,British,British,26584
1,Dutch,Dutch,1055
2,British,Unknown,750
3,British,Dutch,354
4,Dutch,Unknown,200
5,British,Spanish,166
6,British,French,130
7,Dutch,French,86


## Step 1.7 — Export cleaned CSV

In [12]:
print('[1.7] Exporting cleaned CSV ...')
df_clean.to_csv(CLEAN_CSV, index=False)
valid_coords = (~(df_clean['Lat'].isna() | df_clean['Lon'].isna())).mean()
print(f'Saved: {CLEAN_CSV}')
print(f'Total rows:         {len(df_clean):,}')
print(f'Date range:         {df_clean["year_int"].min()}-{df_clean["year_int"].max()}')
print(f'Nations:            {sorted(df_clean["Nationality"].unique())}')
print(f'Valid coordinates:  {valid_coords*100:.1f}%')

[1.7] Exporting cleaned CSV ...
Saved: /run/media/christian/64609cdd-60c1-4856-8535-f3f0c6d7375c/home/christian/OneDrive/DTU/02806 Social Data Analysis and Visualization/winds-of-empire/data/cliwoc_clean.csv
Total rows:         251,456
Date range:         1750-1850
Nations:            ['British', 'Dutch', 'French', 'Spanish']
Valid coordinates:  98.1%


## Done

All output files have been written to `data/`. You can now open `explainer_notebook.ipynb` for the full analysis and visualisations, or open `index.html` to view the website.